# Model Training

Trains 5 XGBoost models (one per prediction target) and logs everything to MLflow.

Targets: `result_encoded` (multi-class), `btts`, `over_2_5`, `over_1_5` (binary), `total_goals` (regression)

In [25]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.xgboost
import pickle
from pathlib import Path
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder

DATA_PATH  = Path('../../datasets/processed/feature_engineered_dataset.csv')
MODELS_DIR = Path('../../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_DATE = '2023-08-01'  # 2023/24 season onwards = test set

params_cls = dict(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42
)

FEATURE_COLS = [
    # categorical (htr_encoded excluded: it is the current match HT result = leakage)
    "home_team_encoded", "away_team_encoded", "referee_encoded",
    "day_encoded", "month_sin", "month_cos",
    # team form
    "home_points_last5", "home_goals_scored_avg5", "home_goals_conceded_avg5",
    "home_sot_avg5", "home_clean_sheets_last5",
    "home_points_last10", "home_goals_scored_avg10", "home_goals_conceded_avg10",
    "home_win_streak",
    "away_points_last5", "away_goals_scored_avg5", "away_goals_conceded_avg5",
    "away_sot_avg5", "away_clean_sheets_last5",
    "away_points_last10", "away_goals_scored_avg10", "away_goals_conceded_avg10",
    "away_win_streak",
    # home/away split
    "home_win_rate_last10", "home_goals_avg_last10",
    "away_win_rate_last10", "away_goals_avg_last10",
    # head-to-head
    "h2h_meetings", "h2h_home_win_rate", "h2h_avg_total_goals", "h2h_btts_rate",
    # shot quality
    "home_conversion_rate_avg5", "home_sot_ratio_avg5",
    "away_conversion_rate_avg5", "away_sot_ratio_avg5",
    # half-time patterns (rolling from previous matches, not the current one)
    "home_2nd_half_goals_avg5", "home_lead_hold_rate", "home_comeback_rate",
    "away_2nd_half_goals_avg5", "away_lead_hold_rate", "away_comeback_rate",
    # referee
    "ref_avg_yellows", "ref_avg_fouls", "ref_home_win_rate",
    # temporal
    "match_week", "season_phase", "home_days_rest", "away_days_rest",
]


## 1. Load Data and Split

In [4]:
df = pd.read_csv(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

train = df[df['Date'] < SPLIT_DATE].copy()
test  = df[df['Date'] >= SPLIT_DATE].copy()

X_train = train[FEATURE_COLS]
X_test  = test[FEATURE_COLS]

print(f'Train: {len(train)} matches ({train["Date"].min().date()} to {train["Date"].max().date()})')
print(f'Test : {len(test)} matches ({test["Date"].min().date()} to {test["Date"].max().date()})')
print(f'Features: {len(FEATURE_COLS)}')

Train: 6855 matches (2005-01-10 to 2023-07-12)
Test : 1036 matches (2023-08-02 to 2026-12-02)
Features: 49


## 2. MLflow Setup and Save Encoders

In [22]:
mlflow.set_tracking_uri('file:../../mlruns')
mlflow.set_registry_uri('sqlite:///../../mlflow.db')
mlflow.set_experiment('soca_scores_match_prediction')

# Fit encoders on full dataset so inference can encode any known team/referee
team_encoder = LabelEncoder()
team_encoder.fit(pd.concat([df['HomeTeam'], df['AwayTeam']]).unique())

referee_encoder = LabelEncoder()
referee_encoder.fit(df['Referee'].unique())

with open(MODELS_DIR / 'team_encoder.pkl', 'wb') as f:
    pickle.dump(team_encoder, f)
with open(MODELS_DIR / 'referee_encoder.pkl', 'wb') as f:
    pickle.dump(referee_encoder, f)

print(f'Teams encoded  : {len(team_encoder.classes_)}')
print(f'Referees encoded: {len(referee_encoder.classes_)}')
print(f'Encoders saved to {MODELS_DIR}')

Teams encoded  : 44
Referees encoded: 71
Encoders saved to ..\..\models


## 3. Result Prediction (Multi-class: Home / Draw / Away)

In [23]:
params_cls = dict(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='mlogloss', random_state=42
)

y_train_r, y_test_r = train['result_encoded'], test['result_encoded']

with mlflow.start_run(run_name='result_prediction'):
    model_result = XGBClassifier(objective='multi:softprob', num_class=3, **params_cls)
    model_result.fit(X_train, y_train_r, eval_set=[(X_test, y_test_r)], verbose=False)

    preds = model_result.predict(X_test)
    acc   = accuracy_score(y_test_r, preds)
    f1    = f1_score(y_test_r, preds, average='weighted')

    mlflow.log_params(params_cls)
    mlflow.log_metric('accuracy', acc)
    mlflow.log_metric('f1_weighted', f1)
    mlflow.xgboost.log_model(model_result, 'model', registered_model_name='soca_result')

    print(f'Result  ï¿½ Accuracy: {acc:.3f} | F1 (weighted): {f1:.3f}')

c:\Users\juliu\OneDrive\Desktop\Projects\Soca-Scores\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [14:06:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026/04/15 14:06:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Result  ï¿½ Accuracy: 0.515 | F1 (weighted): 0.466


Successfully registered model 'soca_result'.
Created version '1' of model 'soca_result'.


## 4. BTTS (Binary)

In [26]:
y_train_b, y_test_b = train['btts'], test['btts']

with mlflow.start_run(run_name='btts_prediction'):
    model_btts = XGBClassifier(objective='binary:logistic', **params_cls)
    model_btts.fit(X_train, y_train_b, eval_set=[(X_test, y_test_b)], verbose=False)

    preds = model_btts.predict(X_test)
    proba = model_btts.predict_proba(X_test)[:, 1]
    acc   = accuracy_score(y_test_b, preds)
    auc   = roc_auc_score(y_test_b, proba)

    mlflow.log_params(params_cls)
    mlflow.log_metric('accuracy', acc)
    mlflow.log_metric('roc_auc', auc)
    mlflow.xgboost.log_model(model_btts, 'model', registered_model_name='soca_btts')

    print(f'BTTS    ï¿½ Accuracy: {acc:.3f} | AUC: {auc:.3f}')

2026/04/15 14:07:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


BTTS    ï¿½ Accuracy: 0.489 | AUC: 0.493


Successfully registered model 'soca_btts'.
Created version '1' of model 'soca_btts'.


## 5. Over 2.5 Goals (Binary)

In [27]:
y_train_o25, y_test_o25 = train['over_2_5'], test['over_2_5']

with mlflow.start_run(run_name='over25_prediction'):
    model_over25 = XGBClassifier(objective='binary:logistic', eval_metric='logloss', **params_cls)
    model_over25.fit(X_train, y_train_o25, eval_set=[(X_test, y_test_o25)], verbose=False)

    preds = model_over25.predict(X_test)
    proba = model_over25.predict_proba(X_test)[:, 1]
    acc   = accuracy_score(y_test_o25, preds)
    auc   = roc_auc_score(y_test_o25, proba)

    mlflow.log_params(params_cls)
    mlflow.log_metric('accuracy', acc)
    mlflow.log_metric('roc_auc', auc)
    mlflow.xgboost.log_model(model_over25, 'model', registered_model_name='soca_over25')

    print(f'Over2.5 ï¿½ Accuracy: {acc:.3f} | AUC: {auc:.3f}')

2026/04/15 14:07:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Over2.5 ï¿½ Accuracy: 0.515 | AUC: 0.538


Successfully registered model 'soca_over25'.
Created version '1' of model 'soca_over25'.


## 6. Over 1.5 Goals (Binary)

In [28]:
y_train_o15, y_test_o15 = train['over_1_5'], test['over_1_5']

with mlflow.start_run(run_name='over15_prediction'):
    model_over15 = XGBClassifier(objective='binary:logistic', eval_metric='logloss', **params_cls)
    model_over15.fit(X_train, y_train_o15, eval_set=[(X_test, y_test_o15)], verbose=False)

    preds = model_over15.predict(X_test)
    proba = model_over15.predict_proba(X_test)[:, 1]
    acc   = accuracy_score(y_test_o15, preds)
    auc   = roc_auc_score(y_test_o15, proba)

    mlflow.log_params(params_cls)
    mlflow.log_metric('accuracy', acc)
    mlflow.log_metric('roc_auc', auc)
    mlflow.xgboost.log_model(model_over15, 'model', registered_model_name='soca_over15')

    print(f'Over1.5 ï¿½ Accuracy: {acc:.3f} | AUC: {auc:.3f}')

2026/04/15 14:07:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Over1.5 ï¿½ Accuracy: 0.813 | AUC: 0.507


Successfully registered model 'soca_over15'.
Created version '1' of model 'soca_over15'.


## 7. Total Goals (Regression)

In [29]:
y_train_g, y_test_g = train['total_goals'], test['total_goals']

params_reg = dict(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective='reg:squarederror', eval_metric='mae', random_state=42
)

with mlflow.start_run(run_name='goals_prediction'):
    model_goals = XGBRegressor(**params_reg)
    model_goals.fit(X_train, y_train_g, eval_set=[(X_test, y_test_g)], verbose=False)

    preds = model_goals.predict(X_test)
    mae   = mean_absolute_error(y_test_g, preds)

    mlflow.log_params(params_reg)
    mlflow.log_metric('mae', mae)
    mlflow.xgboost.log_model(model_goals, 'model', registered_model_name='soca_goals')

    print(f'Goals   ï¿½ MAE: {mae:.3f}')

2026/04/15 14:08:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Goals   ï¿½ MAE: 1.377


Successfully registered model 'soca_goals'.
Created version '1' of model 'soca_goals'.


## 8. Summary

In [30]:
client = mlflow.tracking.MlflowClient()

print('Registered models:')
for rm in client.search_registered_models():
    latest = client.get_latest_versions(rm.name)[0]
    run    = client.get_run(latest.run_id)
    print(f'  {rm.name:20s}  v{latest.version}  metrics={dict(run.data.metrics)}')

print()
print('Launch MLflow UI:')
print('  mlflow ui --backend-store-uri sqlite:///../../mlflow.db')

Registered models:


C:\Users\juliu\AppData\Local\Temp\ipykernel_12628\989265584.py:5: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(rm.name)[0]


  soca_btts             v1  metrics={'accuracy': 0.4893822393822394, 'roc_auc': 0.49284846901131335}
  soca_goals            v1  metrics={'mae': 1.37709379196167}
  soca_over15           v1  metrics={'accuracy': 0.8127413127413128, 'roc_auc': 0.5071094011375042}
  soca_over25           v1  metrics={'accuracy': 0.5154440154440154, 'roc_auc': 0.5381296619432034}
  soca_result           v1  metrics={'accuracy': 0.5154440154440154, 'f1_weighted': 0.4661181257252617}

Launch MLflow UI:
  mlflow ui --backend-store-uri ../../mlruns
